# 재료 대체 기반 레시피 추천: Colab 실행 노트북

`src/`, `training/`의 코드를 한 노트북에 모은 버전입니다. **위에서부터 순서대로** 실행하세요.

| 블록 | 원본 | 셀 실행 시 |
|---|---|---|
| [1] 엔진 | `src/engine.py` | 공통 경로 설정, 모델 정의 |
| [2] 데이터 | `training/data.py` | 그래프 구성 함수 정의 |
| [3] 순위 평가 지표 | `training/ranking_metrics.py` | 지표 함수 정의 |
| [4] 1단계 | `training/stage1.py` | `STAGE1_RUN` 명령 실행 (prepare → train → evaluate → ranking-metrics) |
| [5] 2단계 | `training/stage2.py` | `STAGE2_RUN` 명령 실행 (prepare → train) |
| [6] 추천 | `src/main.py` | 대화형 추천 실행 (`q` 입력 시 종료) |

- 실행 전 Colab 파일 탭에서 `data/`, `checkpoints/`, `results/` 폴더를 `/content`에 바로 올립니다. 경로는 모두 `/content` 기준 상대 경로이고, `artifacts/`는 실행 중 자동으로 만들어집니다.
- 옵션은 각 블록 끝의 `*_ARGS` 값을 바꾸면 됩니다. 이미 만든 결과를 다시 만들지 않으려면 `*_RUN` 목록에서 해당 명령을 주석 처리합니다.
- 노트북에서는 모든 코드가 한 네임스페이스에 있으므로, 이름이 겹치는 1·2단계 명령 함수는 `stage1_train`, `stage2_train`처럼 접두어를 붙였습니다.
- GPU 런타임을 쓰면 BGE 인코딩(prepare)이 빨라집니다.


In [ ]:
!nvidia-smi

In [ ]:
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.11.0+cu130.html

!pip install torch-geometric

In [ ]:
# ======================================================================
# [1] 엔진  (src/engine.py)
# ======================================================================
from __future__ import annotations

from pathlib import Path

# ---------------- 공통 경로 (모든 블록에서 사용, Colab 작업 폴더 /content 기준) ----------------
# data/, checkpoints/, results/ 폴더를 /content에 바로 올려서 사용. artifacts/는 실행 중 자동 생성
RECIPES_CSV = Path("data/stage1/v3/recipes_masked_v3.csv")
PAIRS_CSV = Path("data/stage1/v3/labeled_pairs_masked_v3.csv")
CATEGORY_CSV = Path("data/stage2/ingredients_category.csv")
CONTRIBUTION_CSV = Path("data/stage2/ingredients_contribution.csv")
STAGE1_EMBEDDINGS = Path("artifacts/stage1_embeddings_v3.pt")
STAGE2_EMBEDDINGS = Path("artifacts/stage2_embeddings.pt")
STAGE1_CHECKPOINT = Path("checkpoints/stage1.pt")
STAGE2_CHECKPOINT = Path("checkpoints/stage2.pt")
STAGE1_PREDICTIONS_CSV = Path("results/stage1_predictions.csv")

# ---------------- 엔진 ----------------
# 레시피와 재료 사이의 연결을 평가하는 2단계 GraphSAGE 엔진
#
# Recipe-Recipe 간 연결: SIMILAR(0~1 유사도)
# Recipe-Ingredient 간 연결: CONTRIBUTION (0~1 기여도)
# 학습 시 예측한 연결 점수는 그래프에 추가하지 않음


import csv
import math
from collections import defaultdict
from collections.abc import Sequence
from functools import lru_cache
from pathlib import Path

import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torch_geometric.data import HeteroData
from sentence_transformers import SentenceTransformer

SIMILAR = ("Recipe", "SIMILAR", "Recipe")
CONTRIBUTION = ("Recipe", "CONTRIBUTION", "Ingredient")
REV_CONTRIBUTION = ("Ingredient", "REV_CONTRIBUTION", "Recipe")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _normalize_name(name: str) -> str:
    return name.strip().lower()


@lru_cache(maxsize=None)
def _read_contribution_links(path: str) -> dict[str, frozenset[str]]:
    # 재료 이름 -> 해당 재료가 실제로 들어가는 원본 레시피 ID 집합
    links: dict[str, set[str]] = defaultdict(set)
    with open(path, newline="", encoding="utf-8") as file:
        for row in csv.DictReader(file):
            links[_normalize_name(row["ingredient"])].add(row["recipe_id"].strip())
    return {name: frozenset(ids) for name, ids in links.items()}


def recipes_with_ingredient(
    ingredient_name: str,
    recipe_ids: Sequence[str | int],
    path: str | Path = CONTRIBUTION_CSV,
) -> Tensor:
    """기여도 CSV에서 재료가 실제로 들어가는 레시피를 찾아 그래프 노드 인덱스로 반환.

    recipe_ids[i]는 i번 Recipe 노드의 원본 레시피 ID이며, 그래프에 없는 레시피는 무시한다.
    """
    linked = _read_contribution_links(str(Path(path).resolve())).get(
        _normalize_name(ingredient_name), frozenset()
    )
    return torch.tensor(
        [index for index, rid in enumerate(recipe_ids) if str(rid).strip() in linked],
        dtype=torch.long,
    )


@lru_cache(maxsize=None)
def _read_categories(path: str) -> dict[str, str]:
    # 재료 이름 -> 카테고리
    with open(path, newline="", encoding="utf-8") as file:
        return {
            _normalize_name(row["ingredient"]): row["category"].strip()
            for row in csv.DictReader(file)
        }


def ingredients_in_same_category(
    ingredient_name: str,
    ingredient_names: Sequence[str],
    path: str | Path = CATEGORY_CSV,
) -> Tensor:
    """카테고리 CSV에서 재료의 카테고리를 찾아 같은 카테고리인 재료 노드 인덱스를 반환.

    ingredient_names[j]는 j번 Ingredient 노드의 재료 이름이며, 결과에는 기준 재료도 포함된다.
    카테고리 CSV에 없는 재료 노드는 후보에서 제외한다.
    """
    categories = _read_categories(str(Path(path).resolve()))
    category = categories.get(_normalize_name(ingredient_name))
    if category is None:
        raise ValueError(f"ingredient absent from category CSV: {ingredient_name}")
    return torch.tensor(
        [
            index
            for index, name in enumerate(ingredient_names)
            if categories.get(_normalize_name(name)) == category
        ],
        dtype=torch.long,
    )


class BGETextEncoder:
    # Recipe와 Ingredient 텍스트를 같은 BGE 임베딩 공간으로 변환하는 인코더
    MODEL_NAME = "BAAI/bge-base-en-v1.5"

    def __init__(self):
        self.model = SentenceTransformer(self.MODEL_NAME, device=DEVICE)
        self.embedding_dim = self.model.get_embedding_dimension()

    def encode_texts(
        self,
        texts: list[str],
        batch_size: int = 32,
        show_progress_bar: bool = False,
    ) -> Tensor:
        # 텍스트 목록을 정규화된 float 텐서 [항목 수, 768]로 인코딩
        if batch_size < 1:
            raise ValueError("batch_size는 1 이상이어야 합니다")
        if any(not isinstance(text, str) or not text.strip() for text in texts):
            raise ValueError("모든 입력 텍스트는 비어 있지 않은 문자열이어야 합니다")
        if not texts:
            return torch.empty((0, self.embedding_dim), dtype=torch.float32)

        vectors = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=show_progress_bar,
            convert_to_tensor=True,
            normalize_embeddings=True,
        )
        return vectors.detach().to(device=DEVICE, dtype=torch.float32)


def _check_pairs(pairs: Tensor, source_count: int, target_count: int) -> None:
    if pairs.ndim != 2 or pairs.shape[0] != 2 or pairs.dtype != torch.long:
        raise ValueError("pairs must be a LongTensor with shape [2, number_of_pairs]")
    if pairs.numel() and (
        pairs[0].min() < 0
        or pairs[0].max() >= source_count
        or pairs[1].min() < 0
        or pairs[1].max() >= target_count
    ):
        raise ValueError("pair index is outside the graph")


def find_recipes_by_ingredient(
    data: HeteroData,
    ingredient_id: int,
    include_predicted: bool = False,
) -> Tensor:
    if not 0 <= ingredient_id < data["Ingredient"].num_nodes:
        raise ValueError("ingredient_id is outside the graph")
    edge_store = data[REV_CONTRIBUTION]
    edge_index = edge_store.edge_index
    mask = edge_index[0] == ingredient_id
    if not include_predicted and "is_predicted" in edge_store:
        mask &= ~edge_store.is_predicted.bool()
    return edge_index[1, mask].unique()


class RecipeEncoder(nn.Module):
    # 1단계: SIMILAR로 연결된 레시피에 대해서만 인접 노드 aggregation
    def __init__(self, recipe_features: int, hidden: int = 32):
        super().__init__()
        self.input = nn.Linear(recipe_features, hidden)
        self.conv1 = WeightedSAGEConv(hidden)
        self.conv2 = WeightedSAGEConv(hidden)
        self.logit_scale = nn.Parameter(torch.tensor(2.0))

    def forward(self, data: HeteroData) -> Tensor:
        x = data["Recipe"].x
        edges = data[SIMILAR].edge_index
        weights = data[SIMILAR].edge_weight
        x = F.relu(self.input(x))
        x = F.relu(self.conv1(x, x, edges, weights))
        return F.normalize(self.conv2(x, x, edges, weights), dim=-1)

    def score(self, recipe_embeddings: Tensor, pairs: Tensor) -> Tensor:
        _check_pairs(pairs, len(recipe_embeddings), len(recipe_embeddings))
        left, right = pairs
        return (recipe_embeddings[left] * recipe_embeddings[right]).sum(
            -1
        ) * self.logit_scale.exp().clamp(max=100)


class WeightedSAGEConv(nn.Module):
    # 이웃 메시지에 엣지 가중치를 곱한 뒤 이웃 수로 나누는 GraphSAGE 계층
    def __init__(self, hidden: int):
        super().__init__()
        self.source = nn.Linear(hidden, hidden, bias=False)
        self.target = nn.Linear(hidden, hidden)

    def forward(
        self, source: Tensor, target: Tensor, edges: Tensor, weights: Tensor
    ) -> Tensor:
        neighbors = torch.zeros_like(target)
        counts = target.new_zeros((target.shape[0], 1))
        if edges.numel():
            messages = source[edges[0]] * weights.unsqueeze(-1)
            neighbors.index_add_(0, edges[1], messages)
            counts.index_add_(0, edges[1], torch.ones_like(weights).unsqueeze(-1))
        return self.target(target) + self.source(neighbors / counts.clamp_min(1))


class IngredientLinkEncoder(nn.Module):
    # 2단계: 단일 CONTRIBUTION 관계에서 0~1 기여도를 회귀
    #   - 1단계 인코더에서 고정된 레시피 임베딩 사용
    #   - 연결이 없는 재료도 자기 노드 경로를 통해 재료 특징으로 임베딩 생성

    def __init__(self, ingredient_features: int, hidden: int = 32):
        super().__init__()
        self.input = nn.Linear(ingredient_features, hidden)
        self.conv = WeightedSAGEConv(hidden)
        self.reverse_conv = WeightedSAGEConv(hidden)
        self.recipe_head = nn.Linear(hidden, hidden)
        self.ingredient_head = nn.Linear(hidden, hidden)
        self.logit_scale = nn.Parameter(torch.tensor(2.0))
        self.bias = nn.Parameter(torch.tensor(0.0))

    def encode_ingredients(self, data: HeteroData, recipe_embeddings: Tensor) -> Tensor:
        ingredient_x = F.relu(self.input(data["Ingredient"].x))
        store = data[CONTRIBUTION]
        mask = ~store.is_predicted.bool() if "is_predicted" in store else slice(None)
        return F.normalize(
            self.conv(
                recipe_embeddings,
                ingredient_x,
                store.edge_index[:, mask],
                store.edge_weight[mask],
            ),
            dim=-1,
        )

    def encode_recipes(self, data: HeteroData, recipe_embeddings: Tensor) -> Tensor:
        ingredient_x = F.relu(self.input(data["Ingredient"].x))
        store = data[REV_CONTRIBUTION]
        mask = ~store.is_predicted.bool() if "is_predicted" in store else slice(None)
        neighbor_messages = self.reverse_conv(
            ingredient_x,
            recipe_embeddings,
            store.edge_index[:, mask],
            store.edge_weight[mask],
        )
        return F.normalize(recipe_embeddings + neighbor_messages, dim=-1)

    def score(
        self,
        data: HeteroData,
        recipe_embeddings: Tensor,
        pairs: Tensor,
    ) -> Tensor:
        _check_pairs(pairs, len(recipe_embeddings), data["Ingredient"].num_nodes)
        ingredients = self.encode_ingredients(data, recipe_embeddings)
        recipes = self.encode_recipes(data, recipe_embeddings)
        recipe = F.normalize(self.recipe_head(recipes[pairs[0]]), dim=-1)
        ingredient = F.normalize(self.ingredient_head(ingredients[pairs[1]]), dim=-1)
        return (recipe * ingredient).sum(-1) * self.logit_scale.exp().clamp(
            max=100
        ) + self.bias

    @torch.no_grad()
    def edge_weights(
        self,
        data: HeteroData,
        recipe_embeddings: Tensor,
        pairs: Tensor,
    ) -> Tensor:
        # 실제 연결 여부와 관계없이 edge의 (예상) 가중치를 0~1의 수치로 반환
        return self.score(data, recipe_embeddings, pairs).sigmoid()

    @torch.no_grad()
    def recommend_substitutes(
        self,
        data: HeteroData,
        recipe_embeddings: Tensor,
        recipe_id: int,
        ingredient_id: int,
        top_k: int = 10,
        *,
        original_contribution: float | None = None,
        category_path: str | Path = CATEGORY_CSV,
    ) -> tuple[Tensor, Tensor]:
        """원래 재료와 같은 카테고리의 재료 중 기여도 차이가 작은 순으로 반환

        후보는 카테고리 CSV에서 원래 재료와 같은 카테고리인 재료로 한정하며, 레시피에 이미 연결된 재료는 제외
        기준 기여도는 전달받은 값 -> 실제 연결의 가중치 -> 모델 예측 순으로 사용

        그래프에 data["Ingredient"].ingredient_name[j] (j번 Ingredient 노드의 재료 이름)이 있어야 함
        반환값: (후보 재료 노드 인덱스, 예측 기여도)
        """
        ingredient_names = getattr(data["Ingredient"], "ingredient_name", None)
        if ingredient_names is None:
            raise ValueError('data["Ingredient"].ingredient_name is required')
        if len(ingredient_names) != data["Ingredient"].num_nodes:
            raise ValueError(
                'data["Ingredient"].ingredient_name must have one entry per Ingredient node'
            )
        if not 0 <= recipe_id < len(recipe_embeddings):
            raise ValueError("recipe_id is outside the graph")
        if not 0 <= ingredient_id < data["Ingredient"].num_nodes:
            raise ValueError("ingredient_id is outside the graph")
        if top_k < 1:
            raise ValueError("top_k must be positive")
        if original_contribution is not None and (
            not math.isfinite(original_contribution)
            or not 0 <= original_contribution <= 1
        ):
            raise ValueError("original_contribution must be finite and between 0 and 1")

        device = recipe_embeddings.device
        store = data[CONTRIBUTION]
        edges = store.edge_index
        connected = edges[1, edges[0] == recipe_id].to(device)
        candidates = ingredients_in_same_category(
            ingredient_names[ingredient_id], ingredient_names, category_path
        ).to(device)
        candidates = candidates[
            ~torch.isin(candidates, connected) & (candidates != ingredient_id)
        ]
        if candidates.numel() == 0:
            return candidates, recipe_embeddings.new_empty(0)

        if original_contribution is not None:
            reference = recipe_embeddings.new_tensor(original_contribution)
        else:
            original = (edges[0] == recipe_id) & (edges[1] == ingredient_id)
            if "is_predicted" in store:
                original &= ~store.is_predicted.bool()
            if original.any():
                reference = store.edge_weight[original][0].to(device)
            else:
                pair = torch.tensor([[recipe_id], [ingredient_id]], device=device)
                reference = self.edge_weights(data, recipe_embeddings, pair)[0]

        pairs = torch.stack((torch.full_like(candidates, recipe_id), candidates))
        weights = self.edge_weights(data, recipe_embeddings, pairs)
        positions = (weights - reference).abs().argsort(stable=True)[:top_k]
        return candidates[positions], weights[positions]

    @torch.no_grad()
    def recommend_recipes(
        self,
        data: HeteroData,
        recipe_embeddings: Tensor,
        ingredient_id: int,
        recipe_id: int,
        top_k: int = 10,
        *,
        contribution_path: str | Path = CONTRIBUTION_CSV,
    ) -> tuple[Tensor, Tensor]:
        """ingredient_id 재료가 실제로 들어가는 레시피 중 recipe_id와 유사한 순으로 반환

        후보는 기여도 CSV(data/stage2/ingredients_contribution.csv)에 연결된 레시피로 한정
        1단계 레시피 임베딩에서 학습한 코사인 유사도를 사용하며 기준 레시피는 제외함

        그래프에 노드 번호와 CSV 표기의 대응이 저장되어 있어야 함
            data["Recipe"].raw_id[i]: i번 Recipe 노드의 원본 레시피 ID (CSV의 recipe_id)
            data["Ingredient"].ingredient_name[j]: j번 Ingredient 노드의 재료 이름 (CSV의 ingredient)
        반환값: (후보 레시피 노드 인덱스, 코사인 유사도)
        """
        recipe_ids = getattr(data["Recipe"], "raw_id", None)
        ingredient_names = getattr(data["Ingredient"], "ingredient_name", None)
        if recipe_ids is None or ingredient_names is None:
            raise ValueError(
                'data["Recipe"].raw_id and data["Ingredient"].ingredient_name are required'
            )
        if not 0 <= recipe_id < len(recipe_embeddings):
            raise ValueError("recipe_id is outside the graph")
        if not 0 <= ingredient_id < data["Ingredient"].num_nodes:
            raise ValueError("ingredient_id is outside the graph")
        if top_k < 1:
            raise ValueError("top_k must be positive")
        if len(recipe_ids) != len(recipe_embeddings):
            raise ValueError(
                'data["Recipe"].raw_id must have one entry per Recipe node'
            )
        if len(ingredient_names) != data["Ingredient"].num_nodes:
            raise ValueError(
                'data["Ingredient"].ingredient_name must have one entry per Ingredient node'
            )

        device = recipe_embeddings.device
        candidates = recipes_with_ingredient(
            ingredient_names[ingredient_id], recipe_ids, contribution_path
        ).to(device)
        candidates = candidates[candidates != recipe_id]
        if candidates.numel() == 0:
            return candidates, recipe_embeddings.new_empty(0)

        embeddings = F.normalize(recipe_embeddings, dim=-1)
        similarities = embeddings[candidates] @ embeddings[recipe_id]
        positions = similarities.argsort(descending=True, stable=True)[:top_k]
        return candidates[positions], similarities[positions]

In [ ]:
# ======================================================================
# [2] 데이터  (training/data.py)
# ======================================================================
from __future__ import annotations

# 2단계 데이터: 1단계 임베딩 pt + 2단계 CSV(카테고리·기여도)로 레시피–재료 그래프 구성
#
# - Recipe 노드: 1단계 임베딩 pt의 recipe_ids 순서, SIMILAR 연결은 1단계 학습 분할의 평가 쌍
# - Ingredient 노드: 카테고리 CSV와 기여도 CSV에 등장하는 재료 이름(정규화 후 알파벳순)
# - CONTRIBUTION 연결: 기여도 CSV의 학습 분할만 메시지 전달 그래프에 포함


import csv
import math
import random
from collections.abc import Callable, Sequence
from pathlib import Path

import torch
from torch import Tensor
from torch_geometric.data import HeteroData



# 1단계 SIMILAR 연결 생성 (training/stage1.py에서 옮김, stage1 블록에서도 사용)
def _context_edges(
    rows: list[tuple[int, int, int]],
) -> tuple[torch.Tensor, torch.Tensor]:
    positive = [(a, b, score / 4) for a, b, score in rows if score >= 1]
    if not positive:
        return torch.empty((2, 0), dtype=torch.long), torch.empty(0)
    forward = torch.tensor([(a, b) for a, b, _ in positive], dtype=torch.long).t()
    weights = torch.tensor([weight for _, _, weight in positive], dtype=torch.float32)
    return torch.cat((forward, forward.flip(0)), dim=1), weights.repeat(2)


ContributionRow = tuple[str, str, float]  # (원본 레시피 ID, 재료 이름, 기여도)


def _read_csv(path: str | Path) -> list[dict[str, str]]:
    with Path(path).open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def read_ingredients(
    category_path: str | Path = CATEGORY_CSV,
    contribution_path: str | Path = CONTRIBUTION_CSV,
) -> tuple[list[str], list[str]]:
    """재료 노드 목록(이름, 카테고리)을 반환. 카테고리 CSV에 없는 재료의 카테고리는 빈 문자열"""
    categories: dict[str, str] = {}
    for row in _read_csv(category_path):
        name, category = _normalize_name(row["ingredient"]), row["category"].strip()
        if not name or not category:
            raise ValueError(f"empty ingredient or category in {category_path}")
        if categories.setdefault(name, category) != category:
            raise ValueError(f"ingredient has two categories: {name}")
    names = set(categories)
    names.update(
        _normalize_name(row["ingredient"]) for row in _read_csv(contribution_path)
    )
    ordered = sorted(names)
    return ordered, [categories.get(name, "") for name in ordered]


def ingredient_text(name: str, category: str) -> str:
    if not category:
        return f"Ingredient: {name}."
    return f"Ingredient: {name}. Category: {category.replace('_', ' ')}."


def read_contributions(path: str | Path = CONTRIBUTION_CSV) -> list[ContributionRow]:
    rows: list[ContributionRow] = []
    seen = set()
    for row in _read_csv(path):
        recipe_id = row["recipe_id"].strip()
        name = _normalize_name(row["ingredient"])
        try:
            weight = float(row["contribution"])
        except ValueError:
            raise ValueError(f"contribution must be a number: {row['contribution']}")
        if not recipe_id or not name:
            raise ValueError("contribution row needs a recipe_id and an ingredient")
        if not math.isfinite(weight) or not 0 <= weight <= 1:
            raise ValueError(f"contribution must be between 0 and 1: {weight}")
        if (recipe_id, name) in seen:
            raise ValueError(f"duplicate contribution pair: {recipe_id}, {name}")
        seen.add((recipe_id, name))
        rows.append((recipe_id, name, weight))
    return rows


def split_contributions(
    rows: Sequence[ContributionRow], seed: int = 42, test_ratio: float = 0.15
) -> dict[str, list[ContributionRow]]:
    if not 0 < test_ratio < 1:
        raise ValueError("test_ratio must be between 0 and 1")
    shuffled = list(rows)
    random.Random(seed).shuffle(shuffled)
    n_test = round(len(shuffled) * test_ratio)
    return {"train": shuffled[n_test:], "test": shuffled[:n_test]}


def prepare_stage2(
    encode: Callable[[list[str]], Tensor],
    *,
    category_path: str | Path = CATEGORY_CSV,
    contribution_path: str | Path = CONTRIBUTION_CSV,
    seed: int = 42,
    test_ratio: float = 0.15,
) -> dict:
    """재료 텍스트 임베딩과 기여도 학습/테스트 분할을 담은 2단계 pt 내용을 생성"""
    names, categories = read_ingredients(category_path, contribution_path)
    features = encode(
        [ingredient_text(name, category) for name, category in zip(names, categories)]
    )
    splits = split_contributions(
        read_contributions(contribution_path), seed, test_ratio
    )
    return {
        "ingredient_names": names,
        "ingredient_categories": categories,
        "ingredient_features": features.detach().cpu().float(),
        "splits": splits,
        "seed": seed,
    }


def set_contributions(
    data: HeteroData,
    pairs: Tensor,
    weights: Tensor,
    is_predicted: Tensor | None = None,
) -> None:
    """CONTRIBUTION과 역방향 REV_CONTRIBUTION을 같은 연결로 설정"""
    data[CONTRIBUTION].edge_index = pairs
    data[CONTRIBUTION].edge_weight = weights
    data[REV_CONTRIBUTION].edge_index = pairs.flip(0)
    data[REV_CONTRIBUTION].edge_weight = weights.clone()
    for store in (data[CONTRIBUTION], data[REV_CONTRIBUTION]):
        if is_predicted is not None:
            store.is_predicted = is_predicted.clone()
        elif "is_predicted" in store:
            del store.is_predicted


def build_graph(
    recipe_ids: Sequence[str | int],
    recipe_features: Tensor,
    similar_rows: Sequence[tuple[int, int, int]],
    ingredient_names: Sequence[str],
    ingredient_features: Tensor,
    contribution_pairs: Tensor,
    contribution_weights: Tensor,
) -> HeteroData:
    """노드 인덱스 기준 입력으로 2단계 그래프를 생성

    similar_rows: 1단계 분할 형식 (레시피 a, 레시피 b, 0~4점). 1점 이상만 양방향 연결
    contribution_pairs: [2, 연결 수] (레시피 노드, 재료 노드), contribution_weights: 0~1
    """
    for features in (recipe_features, ingredient_features):
        if features.ndim != 2 or not torch.isfinite(features).all():
            raise ValueError("node features must be finite 2D tensors")
    if len(recipe_ids) != len(recipe_features):
        raise ValueError("recipe_ids must have one entry per recipe feature row")
    if len(ingredient_names) != len(ingredient_features):
        raise ValueError(
            "ingredient_names must have one entry per ingredient feature row"
        )
    recipe_count, ingredient_count = len(recipe_features), len(ingredient_features)
    if any(
        not (0 <= a < recipe_count and 0 <= b < recipe_count)
        for a, b, _ in similar_rows
    ):
        raise ValueError("out-of-range graph edge in SIMILAR")
    if contribution_pairs.numel() and (
        contribution_pairs.min() < 0
        or contribution_pairs[0].max() >= recipe_count
        or contribution_pairs[1].max() >= ingredient_count
    ):
        raise ValueError("out-of-range graph edge in CONTRIBUTION")
    if contribution_weights.numel() and (
        not torch.isfinite(contribution_weights).all()
        or contribution_weights.min() < 0
        or contribution_weights.max() > 1
    ):
        raise ValueError("contribution must be a number between 0 and 1")

    data = HeteroData()
    data["Recipe"].x = recipe_features.float()
    data["Recipe"].raw_id = [str(value) for value in recipe_ids]
    data["Ingredient"].x = ingredient_features.float()
    data["Ingredient"].ingredient_name = list(ingredient_names)
    data[SIMILAR].edge_index, data[SIMILAR].edge_weight = _context_edges(
        list(similar_rows)
    )
    set_contributions(data, contribution_pairs, contribution_weights.float())
    return data


def _index_rows(
    rows: Sequence[ContributionRow],
    recipe_index: dict[str, int],
    ingredient_index: dict[str, int],
) -> tuple[Tensor, Tensor]:
    missing = sorted({recipe for recipe, _, _ in rows if recipe not in recipe_index})
    if missing:
        raise ValueError(
            f"contribution recipes absent from stage-1 embeddings: {', '.join(missing[:10])}"
        )
    unknown = sorted({name for _, name, _ in rows if name not in ingredient_index})
    if unknown:
        raise ValueError(
            f"contribution ingredients absent from stage-2 embeddings: {', '.join(unknown[:10])}"
        )
    if not rows:
        return torch.empty((2, 0), dtype=torch.long), torch.empty(0)
    pairs = (
        torch.tensor(
            [
                (recipe_index[recipe], ingredient_index[name])
                for recipe, name, _ in rows
            ],
            dtype=torch.long,
        )
        .t()
        .contiguous()
    )
    return pairs, torch.tensor([weight for _, _, weight in rows], dtype=torch.float)


def load_dataset(
    stage1_path: str | Path = STAGE1_EMBEDDINGS,
    stage2_path: str | Path = STAGE2_EMBEDDINGS,
) -> tuple[HeteroData, dict]:
    """두 임베딩 pt로 그래프와 기여도 학습/테스트 대상을 생성

    메시지 전달 그래프에는 학습 분할의 기여도 연결만 들어가며, 테스트 연결은 포함되지 않는다.
    """
    stage1 = torch.load(stage1_path, map_location="cpu", weights_only=True)
    stage2 = torch.load(stage2_path, map_location="cpu", weights_only=True)
    recipe_ids = [str(value) for value in stage1["recipe_ids"]]
    names = list(stage2["ingredient_names"])
    recipe_index = {value: index for index, value in enumerate(recipe_ids)}
    ingredient_index = {name: index for index, name in enumerate(names)}

    targets = {}
    for split in ("train", "test"):
        pairs, labels = _index_rows(
            stage2["splits"][split], recipe_index, ingredient_index
        )
        targets[split] = {"ingredient": (pairs, labels)}
    train_keys = set(map(tuple, targets["train"]["ingredient"][0].t().tolist()))
    if any(
        tuple(pair) in train_keys
        for pair in targets["test"]["ingredient"][0].t().tolist()
    ):
        raise ValueError("stage2 test target appears in message-passing graph")

    data = build_graph(
        recipe_ids,
        stage1["recipe_features"],
        stage1["splits"]["train"],
        names,
        stage2["ingredient_features"],
        *targets["train"]["ingredient"],
    )
    return data, targets

In [ ]:
# ======================================================================
# [3] 순위 평가 지표  (training/ranking_metrics.py)
# ======================================================================
from __future__ import annotations

# 레시피 유사도 예측의 순위 평가: NDCG@K, Precision@K, Recall@K, Spearman's rho
# 동점 레시피는 순서를 정할 수 없으므로 동점 내 모든 순서의 기댓값으로 계산 (파일 순서에 영향받지 않음)

import argparse
import csv
import math
from collections import defaultdict
from itertools import groupby
from pathlib import Path

DEFAULT_KS = (10, 50, 100)
DEFAULT_THRESHOLD = 2.0


def _validate(scores: list[float], relevance: list[float], k: int = 1) -> None:
    if not scores or len(scores) != len(relevance):
        raise ValueError("scores and relevance must be non-empty and of equal length")
    if not all(map(math.isfinite, scores)) or not all(map(math.isfinite, relevance)):
        raise ValueError("scores and relevance must be finite")
    if k < 1:
        raise ValueError("k must be at least 1")


def _tie_groups(scores: list[float]) -> list[list[int]]:
    order = sorted(range(len(scores)), key=lambda index: -scores[index])
    return [list(group) for _, group in groupby(order, key=lambda index: scores[index])]


def _expected_relevant_in_top_k(
    scores: list[float], relevance: list[float], k: int, threshold: float
) -> float:
    hits = 0.0
    position = 0
    for group in _tie_groups(scores):
        end = position + len(group)
        share = max(0, min(end, k) - position) / len(group)
        hits += share * sum(relevance[index] >= threshold for index in group)
        position = end
    return hits


def ndcg_at_k(scores: list[float], relevance: list[float], k: int) -> float:
    """정렬이 적절한지. 정답 점수를 그대로 이득으로 사용 (모든 정답이 0이면 nan)"""
    _validate(scores, relevance, k)
    ideal = sum(
        gain / math.log2(rank + 2)
        for rank, gain in enumerate(sorted(relevance, reverse=True)[:k])
    )
    if ideal == 0:
        return float("nan")
    dcg = 0.0
    position = 0
    for group in _tie_groups(scores):
        end = position + len(group)
        gain = sum(relevance[index] for index in group) / len(group)
        dcg += gain * sum(1 / math.log2(rank + 2) for rank in range(position, min(end, k)))
        position = end
    return dcg / ideal


def precision_at_k(
    scores: list[float],
    relevance: list[float],
    k: int,
    threshold: float = DEFAULT_THRESHOLD,
) -> float:
    """상위 K개 중 적절한 레시피 비율. 후보가 K개보다 적으면 후보 수로 나눔"""
    _validate(scores, relevance, k)
    hits = _expected_relevant_in_top_k(scores, relevance, k, threshold)
    return hits / min(k, len(scores))


def recall_at_k(
    scores: list[float],
    relevance: list[float],
    k: int,
    threshold: float = DEFAULT_THRESHOLD,
) -> float:
    """전체 적절한 레시피 중 상위 K개에서 찾은 비율 (적절한 레시피가 없으면 nan)"""
    _validate(scores, relevance, k)
    total = sum(value >= threshold for value in relevance)
    if total == 0:
        return float("nan")
    return _expected_relevant_in_top_k(scores, relevance, k, threshold) / total


def _average_ranks(values: list[float]) -> list[float]:
    ranks = [0.0] * len(values)
    position = 0
    for group in _tie_groups(values):
        for index in group:
            ranks[index] = position + (len(group) - 1) / 2
        position += len(group)
    return ranks


def spearman_rho(scores: list[float], relevance: list[float]) -> float:
    """절대값을 제외한 순위 상관. 후보가 2개 미만이거나 한쪽 값이 모두 같으면 nan"""
    _validate(scores, relevance)
    x = _average_ranks(scores)
    y = _average_ranks(relevance)
    mean = (len(x) - 1) / 2
    covariance = sum((a - mean) * (b - mean) for a, b in zip(x, y))
    x_variance = sum((a - mean) ** 2 for a in x)
    y_variance = sum((b - mean) ** 2 for b in y)
    if x_variance == 0 or y_variance == 0:
        return float("nan")
    return covariance / math.sqrt(x_variance * y_variance)


def ranking_metrics(
    scores: list[float],
    relevance: list[float],
    ks: tuple[int, ...] = DEFAULT_KS,
    threshold: float = DEFAULT_THRESHOLD,
) -> dict[str, float]:
    metrics = {"spearman": spearman_rho(scores, relevance)}
    for k in ks:
        metrics[f"ndcg@{k}"] = ndcg_at_k(scores, relevance, k)
        metrics[f"precision@{k}"] = precision_at_k(scores, relevance, k, threshold)
        metrics[f"recall@{k}"] = recall_at_k(scores, relevance, k, threshold)
    return metrics


def mean_ranking_metrics(
    candidates: list[tuple[list[float], list[float]]],
    ks: tuple[int, ...] = DEFAULT_KS,
    threshold: float = DEFAULT_THRESHOLD,
) -> dict[str, float]:
    """기준 레시피별 지표의 평균. 값을 정의할 수 없는(nan) 기준 레시피는 지표별로 제외"""
    collected = defaultdict(list)
    for scores, relevance in candidates:
        for name, value in ranking_metrics(scores, relevance, ks, threshold).items():
            if not math.isnan(value):
                collected[name].append(value)
    return {
        name: sum(values) / len(values) if values else float("nan")
        for name, values in collected.items()
    }


def write_metrics_csv(path: Path, results: dict[str, dict[str, float]]) -> None:
    """분할별 지표를 split,metric,k,value 형식으로 저장 (Spearman's rho는 k가 비어 있음)"""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, lineterminator="\n")
        writer.writerow(("split", "metric", "k", "value"))
        for split, metrics in results.items():
            for name, value in metrics.items():
                metric, _, k = name.partition("@")
                writer.writerow((split, metric, k, f"{value:.4f}"))


def _candidates_by_recipe(
    rows: list[dict[str, str]], score_column: str
) -> list[tuple[list[float], list[float]]]:
    # 유사도는 대칭이므로 한 쌍을 양쪽 레시피의 후보로 모두 사용
    grouped = defaultdict(lambda: ([], []))
    for row in rows:
        for recipe_id in (row["a_id"], row["b_id"]):
            grouped[recipe_id][0].append(float(row[score_column]))
            grouped[recipe_id][1].append(float(row["similarity"]))
    return [group for group in grouped.values() if len(group[0]) >= 2]


def _print_metrics(metrics: dict[str, float], ks: tuple[int, ...], indent: str = "") -> None:
    print(f"{indent}Spearman's rho={metrics['spearman']:.3f}")
    for k in ks:
        print(
            f"{indent}K={k}: NDCG={metrics[f'ndcg@{k}']:.3f} "
            f"Precision={metrics[f'precision@{k}']:.3f} "
            f"Recall={metrics[f'recall@{k}']:.3f}"
        )


def ranking_metrics_command(args: argparse.Namespace) -> None:
    """예측 결과 CSV의 순위 평가 지표 출력 (stage1 블록의 "ranking-metrics" 명령)"""
    with args.predictions.open(encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    scores = [float(row[args.score_column]) for row in rows]
    relevance = [float(row["similarity"]) for row in rows]
    ks = tuple(args.k)
    print(f"쌍={len(rows)} 고유 예측 점수={len(set(scores))}개")
    if args.group_by == "recipe":
        candidates = _candidates_by_recipe(rows, args.score_column)
        print(f"기준 레시피(후보 2개 이상)={len(candidates)}개의 평균")
        _print_metrics(mean_ranking_metrics(candidates, ks, args.threshold), ks)
    else:
        _print_metrics(ranking_metrics(scores, relevance, ks, args.threshold), ks)

In [ ]:
# ======================================================================
# [4] 1단계: 레시피 유사도 학습  (training/stage1.py)
# ======================================================================
from __future__ import annotations

# 전체 레시피 CSV와 평가 쌍 기반 1단계 유사도 학습
#     - 실행 순서: prepare → train → evaluate


import argparse
import ast
import csv
import json
import math
import random
from collections import Counter
from pathlib import Path

import torch


def _operation_text(data: dict) -> str:
    clauses = data.get("clauses") or []
    if data.get("raw"):
        text = data["raw"].strip()
    else:
        # 조건만 담은 OP의 verb는 if/until 등을 한 번 더 포함함
        clause_markers = {clause.get("marker") for clause in clauses}
        text = " ".join(
            data[key].strip()
            for key in ("subject", "verb", "object", "location", "other")
            if data.get(key)
            and not (key == "subject" and data[key].lower() == "you")
            and not (key == "verb" and data[key] in clause_markers)
        )
    for key, prefix in (("time", "for"), ("temperature", "at")):
        value = data.get(key)
        if value and value.lower() not in text.lower():
            detail = (
                value if value.lower().startswith(prefix + " ") else f"{prefix} {value}"
            )
            text = " ".join(filter(None, (text, detail)))
    for clause in clauses:
        nested = " and ".join(
            filter(None, (_operation_text(op) for op in clause.get("ops", [])))
        )
        body = " ".join(filter(None, (clause.get("text"), nested)))
        if body:
            condition = " ".join(filter(None, (clause.get("marker"), body)))
            text = ", ".join(filter(None, (text, condition)))
    return text


def _step_text(step: dict) -> str:
    parts = []
    for operation in step.get("operations") or []:
        text = _operation_text(operation["data"])
        if text:
            # 별도 OP로 분리된 부정어는 다음 동작과 붙여서 유지
            if parts and parts[-1] == "not":
                parts[-1] = f"not {text}"
            else:
                parts.append(text)
    text = "; ".join(parts) or (step.get("text") or "").strip()
    if not text:
        raise ValueError("structured step has no operation or text")
    return text if text.endswith((".", "!", "?")) else text + "."


def _items(value: str) -> list[str]:
    try:
        parsed = json.loads(value)
    except json.JSONDecodeError:
        # 기존 CSV는 Python 문자열 목록, 구조화된 steps는 JSON으로 저장됨
        parsed = ast.literal_eval(value)
    if not isinstance(parsed, list) or not all(
        isinstance(item, (str, dict)) for item in parsed
    ):
        raise ValueError("expected a list of strings or step objects in recipe CSV")
    # 구조화된 steps는 JSON 문법 대신 조리 동작과 조건을 입력으로 사용.
    return [item if isinstance(item, str) else _step_text(item) for item in parsed]


def recipe_text(row: dict[str, str]) -> str:
    steps = " ".join(_items(row["steps"]))
    return f"Recipe: {row['name']}. Steps: {steps}"


def unique_recipes(rows: list[dict[str, str]]) -> list[dict[str, str]]:
    # 같은 ID의 중복 행: 마스킹된 조리 단계의 변형 가능성
    # ID별 특징 벡터 하나를 위해 첫 번째 행 사용
    by_id = {}
    for row in rows:
        by_id.setdefault(row["id"], row)
    return list(by_id.values())


def split_pairs(
    rows: list[dict[str, str]], id_to_index: dict[str, int], seed: int
) -> dict[str, list[tuple[int, int, int]]]:
    # 0~4점 비율 유지, 학습/검증/테스트 70/15/15 분할
    buckets: dict[int, list[tuple[int, int, int]]] = {score: [] for score in range(5)}
    seen = set()
    for row in rows:
        left, right = row["a_id"], row["b_id"]
        if left not in id_to_index or right not in id_to_index:
            raise ValueError(f"pair references a missing recipe: {left}, {right}")
        if left == right:
            raise ValueError(f"self pair: {left}")
        key = tuple(sorted((left, right)))
        if key in seen:
            raise ValueError(f"duplicate recipe pair: {left}, {right}")
        seen.add(key)
        score = int(row["similarity"])
        if score not in buckets:
            raise ValueError(f"similarity must be an integer from 0 to 4: {score}")
        buckets[score].append((id_to_index[left], id_to_index[right], score))

    rng = random.Random(seed)
    result: dict[str, list[tuple[int, int, int]]] = {
        name: [] for name in ("train", "val", "test")
    }
    for bucket in buckets.values():
        rng.shuffle(bucket)
        n_test = round(len(bucket) * 0.15)
        n_val = round(len(bucket) * 0.15)
        result["test"].extend(bucket[:n_test])
        result["val"].extend(bucket[n_test : n_test + n_val])
        result["train"].extend(bucket[n_test + n_val :])
    for split in result.values():
        rng.shuffle(split)
    return result


def _pairs_and_labels(
    rows: list[tuple[int, int, int]],
) -> tuple[torch.Tensor, torch.Tensor]:
    pairs = (
        torch.tensor([(a, b) for a, b, _ in rows], dtype=torch.long).t().contiguous()
    )
    labels = torch.tensor([score / 4 for _, _, score in rows], dtype=torch.float32)
    return pairs, labels


def _fold_context(
    rows: list[tuple[int, int, int]], selected: list[int]
) -> list[tuple[int, int, int]]:
    excluded = set(selected)
    return [row for index, row in enumerate(rows) if index not in excluded]


def stage1_prepare(args: argparse.Namespace) -> None:

    recipes = unique_recipes(_read_csv(args.recipes))
    id_to_index = {row["id"]: index for index, row in enumerate(recipes)}
    splits = split_pairs(_read_csv(args.pairs), id_to_index, args.seed)
    encoder = BGETextEncoder()
    texts = [recipe_text(row) for row in recipes]
    features = encoder.encode_texts(
        texts, batch_size=args.batch_size, show_progress_bar=True
    ).cpu()
    artifact = {
        "recipe_ids": [row["id"] for row in recipes],
        "recipe_features": features,
        "splits": splits,
        "seed": args.seed,
        "encoder": encoder.MODEL_NAME,
    }
    args.output.parent.mkdir(parents=True, exist_ok=True)
    torch.save(artifact, args.output)
    print(
        f"saved {len(recipes)} recipes and {sum(map(len, splits.values()))} pairs to {args.output}"
    )
    for name, split in splits.items():
        print(
            f"{name}: {len(split)} pairs; scores {dict(sorted(Counter(row[2] for row in split).items()))}"
        )


def _graph(features: torch.Tensor, context_rows: list[tuple[int, int, int]]):
    from torch_geometric.data import HeteroData

    data = HeteroData()
    data["Recipe"].x = features
    data[SIMILAR].edge_index, data[SIMILAR].edge_weight = _context_edges(context_rows)
    return data.to(features.device)  # 연결 텐서를 특징과 같은 장치(GPU/CPU)로


def _load_artifact(path: Path) -> dict:
    artifact = torch.load(path, map_location="cpu", weights_only=True)
    if not isinstance(artifact["recipe_features"], torch.Tensor):
        raise ValueError("invalid stage-1 feature artifact")
    return artifact


def stage1_train(args: argparse.Namespace) -> None:

    artifact = _load_artifact(args.data)
    features = artifact["recipe_features"].to(DEVICE)
    train_rows = artifact["splits"]["train"]
    val_rows = artifact["splits"]["val"]
    if not train_rows:
        raise ValueError("training split is empty")
    if not val_rows:
        raise ValueError(
            "validation split is empty; early stopping requires validation pairs"
        )
    if args.folds < 2 or args.epochs < 1 or args.patience < 1:
        raise ValueError("folds must be >= 2; epochs and patience must be >= 1")
    rng = random.Random(args.seed)
    model = RecipeEncoder(features.shape[1], args.hidden).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    val_data = _graph(features, train_rows)
    val_pairs, val_labels = _pairs_and_labels(val_rows)
    val_pairs, val_labels = val_pairs.to(DEVICE), val_labels.to(DEVICE)
    best_val_rmse = float("inf")
    best_epoch = 0
    stale_epochs = 0

    # 폴드 교대 제외로 모든 학습 쌍을 예측 대상에 포함
    # 현재 예측 폴드의 SIMILAR 연결은 입력 그래프에서 제외
    for epoch in range(args.epochs):
        order = list(range(len(train_rows)))
        rng.shuffle(order)
        fold_indices = [order[i :: args.folds] for i in range(args.folds)]
        squared_error_sum = 0.0
        prediction_count = 0
        model.train()
        for selected in fold_indices:
            if not selected:
                continue
            context = _fold_context(train_rows, selected)
            targets = [train_rows[i] for i in selected]
            pairs, labels = _pairs_and_labels(targets)
            pairs, labels = pairs.to(DEVICE), labels.to(DEVICE)
            data = _graph(features, context)
            optimizer.zero_grad()
            predictions = model.score(model(data), pairs).sigmoid()
            # 학습 손실과 출력 오차를 모두 정규화된 0~1 척도 RMSE로 계산
            squared_errors = (predictions - labels).square()
            loss = (squared_errors.mean() + 1e-8).sqrt()
            loss.backward()
            optimizer.step()
            squared_error_sum += float(squared_errors.detach().sum())
            prediction_count += len(labels)
        model.eval()
        with torch.no_grad():
            val_predictions = model.score(model(val_data), val_pairs).sigmoid()
            val_rmse = float((val_predictions - val_labels).square().mean().sqrt())
        if not math.isfinite(val_rmse):
            raise ValueError("validation RMSE is not finite")
        train_rmse = (squared_error_sum / prediction_count) ** 0.5
        print(f"epoch={epoch + 1} train_rmse={train_rmse:.4f} val_rmse={val_rmse:.4f}")

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch + 1
            stale_epochs = 0
            args.output.parent.mkdir(parents=True, exist_ok=True)
            torch.save(
                {
                    # GPU가 없는 환경에서도 읽을 수 있도록 CPU 텐서로 저장
                    "model": {k: v.cpu() for k, v in model.state_dict().items()},
                    "hidden": args.hidden,
                    "recipe_features": features.shape[1],
                    "data_seed": artifact["seed"],
                    "best_epoch": best_epoch,
                    "best_val_rmse": best_val_rmse,
                },
                args.output,
            )
        else:
            stale_epochs += 1
            if stale_epochs >= args.patience:
                print(f"early_stopping epoch={epoch + 1} patience={args.patience}")
                break
    print(
        f"best_epoch={best_epoch} best_val_rmse={best_val_rmse:.4f} checkpoint={args.output}"
    )


def score_metrics(predictions: torch.Tensor, labels: torch.Tensor) -> dict:
    if predictions.ndim != 1 or labels.shape != predictions.shape or not labels.numel():
        raise ValueError(
            "predictions and labels must be non-empty 1D tensors of equal length"
        )
    if not torch.isfinite(predictions).all() or not torch.isfinite(labels).all():
        raise ValueError("predictions and labels must be finite")

    actual = (labels * 4).round().long().clamp(0, 4)
    predicted = (predictions * 4).round().long().clamp(0, 4)
    errors = predictions - labels
    exact = (predicted == actual).float().mean().item()
    within_one = ((predicted - actual).abs() <= 1).float().mean().item()
    positive_actual = actual >= 2
    positive_predicted = predicted >= 2
    tp = int((positive_actual & positive_predicted).sum())
    fp = int((~positive_actual & positive_predicted).sum())
    fn = int((positive_actual & ~positive_predicted).sum())
    tn = int((~positive_actual & ~positive_predicted).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    specificity = tn / (tn + fp) if tn + fp else 0.0
    confusion = [
        [int(((actual == truth) & (predicted == guess)).sum()) for guess in range(5)]
        for truth in range(5)
    ]
    return {
        "mae": errors.mean().item() * 4,
        "rmse": errors.square().mean().sqrt().item(),
        "exact_accuracy": exact,
        "within_one_accuracy": within_one,
        "zero_baseline_exact": (actual == 0).float().mean().item(),
        "zero_baseline_within_one": (actual <= 1).float().mean().item(),
        "positive_precision": precision,
        "positive_recall": recall,
        "positive_f1": f1,
        "balanced_accuracy": (recall + specificity) / 2,
        "confusion": confusion,
    }


def _write_prediction_rows(
    writer: csv.writer,
    recipe_ids: list[str],
    rows: list[tuple[int, int, int]],
    predictions: torch.Tensor,
) -> None:
    predicted_scores = (predictions * 4).round().long().clamp(0, 4).tolist()
    for (a, b, score), predicted_score, similarity in zip(
        rows, predicted_scores, predictions.tolist(), strict=True
    ):
        writer.writerow(
            (recipe_ids[a], recipe_ids[b], score, predicted_score, similarity)
        )


def stage1_evaluate(args: argparse.Namespace) -> None:

    artifact = _load_artifact(args.data)
    features = artifact["recipe_features"].to(DEVICE)
    saved = torch.load(args.checkpoint, map_location="cpu", weights_only=True)
    if (
        saved["recipe_features"] != features.shape[1]
        or saved["data_seed"] != artifact["seed"]
    ):
        raise ValueError("checkpoint does not match the feature artifact")
    model = RecipeEncoder(features.shape[1], saved["hidden"]).to(DEVICE)
    model.load_state_dict(saved["model"])
    model.eval()
    # 입력 그래프에서 검증·테스트 연결 제외
    # 직접 연결되지 않은 쌍도 레시피별 텍스트 특징으로 점수 계산
    data = _graph(features, artifact["splits"]["train"])
    args.output.parent.mkdir(parents=True, exist_ok=True)
    print("변환 기준: 예측 유사도 × 4를 반올림해 0~4점으로 변환, 2점 이상은 유사 쌍")
    with torch.no_grad(), args.output.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(
            (
                "a_id",
                "b_id",
                "similarity",
                "predicted_similarity",
                "predicted_similarity_0_1",
            )
        )
        embeddings = model(data)
        for split_name in ("test",):
            rows = artifact["splits"][split_name]
            pairs, labels = _pairs_and_labels(rows)
            pairs, labels = pairs.to(DEVICE), labels.to(DEVICE)
            predictions = model.score(embeddings, pairs).sigmoid()
            metrics = score_metrics(predictions, labels)
            _write_prediction_rows(writer, artifact["recipe_ids"], rows, predictions)
            print(f"{split_name}: n={len(rows)}")
            print(
                f"  점수 오차(0~4 라벨 기준): MAE={metrics['mae']:.4f} "
                f"  RMSE={metrics['rmse']*4:.4f}"
            )
            print(f"  유사도 오차(0~1): RMSE={metrics['rmse']:.4f}")
            print(f"  정답 비율={metrics['exact_accuracy']:.1%}")
            print(f"  근접 비율(오차 1점 이내)={metrics['within_one_accuracy']:.1%}")
            print(
                f"  2점 이상(positive) 탐지: Precision={metrics['positive_precision']:.1%} "
                f"Recall={metrics['positive_recall']:.1%} "
                f"F1={metrics['positive_f1']:.3f} "
                f"균형 정확도={metrics['balanced_accuracy']:.1%}"
            )
            print("  혼동 행렬(행=실제 점수, 열=예측 점수(0~4)):")
            for score, counts in enumerate(metrics["confusion"]):
                print(f"    {score}: {counts}")
    print(f"predictions={args.output}")


def stage1_predict(args: argparse.Namespace) -> None:

    artifact = _load_artifact(args.data)
    ids = {
        str(recipe_id): index for index, recipe_id in enumerate(artifact["recipe_ids"])
    }
    a_id, b_id = str(args.a_id).strip(), str(args.b_id).strip()
    missing = [recipe_id for recipe_id in (a_id, b_id) if recipe_id not in ids]
    if missing:
        raise ValueError(f"recipe ID absent from stage1.pt: {', '.join(missing)}")
    features = artifact["recipe_features"].to(DEVICE)
    saved = torch.load(args.checkpoint, map_location="cpu", weights_only=True)
    if (
        saved["recipe_features"] != features.shape[1]
        or saved["data_seed"] != artifact["seed"]
    ):
        raise ValueError("checkpoint does not match the feature artifact")
    model = RecipeEncoder(features.shape[1], saved["hidden"]).to(DEVICE)
    model.load_state_dict(saved["model"])
    model.eval()
    data = _graph(features, artifact["splits"]["train"])
    pair = torch.tensor([[ids[a_id]], [ids[b_id]]], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        similarity = model.score(model(data), pair).sigmoid().item()
    print(
        f"{args.a_id},{args.b_id}: predicted_similarity={similarity:.4f} (scale 0..1)"
    )


def rank_similar_recipes(
    model, embeddings: torch.Tensor, recipe_index: int, top_k: int
) -> tuple[torch.Tensor, torch.Tensor]:
    if not 0 <= recipe_index < len(embeddings):
        raise ValueError("recipe index is outside the graph")
    if top_k < 1:
        raise ValueError("top_k must be positive")
    candidates = torch.arange(len(embeddings), device=embeddings.device)
    candidates = candidates[candidates != recipe_index]
    if candidates.numel() == 0:
        return candidates, embeddings.new_empty(0)
    pairs = torch.stack((torch.full_like(candidates, recipe_index), candidates))
    similarities = model.score(embeddings, pairs).sigmoid()
    values, positions = similarities.topk(min(top_k, len(candidates)))
    return candidates[positions], values


def stage1_top_similar(args: argparse.Namespace) -> None:

    artifact = _load_artifact(args.data)
    recipe_ids = [str(recipe_id) for recipe_id in artifact["recipe_ids"]]
    recipe_id = str(args.recipe_id).strip()
    id_to_index = {value: index for index, value in enumerate(recipe_ids)}
    if recipe_id not in id_to_index:
        raise ValueError(f"recipe ID absent from stage1.pt: {recipe_id}")
    if args.top_k < 1:
        raise ValueError("top_k must be positive")

    features = artifact["recipe_features"].to(DEVICE)
    saved = torch.load(args.checkpoint, map_location="cpu", weights_only=True)
    if (
        saved["recipe_features"] != features.shape[1]
        or saved["data_seed"] != artifact["seed"]
    ):
        raise ValueError("checkpoint does not match the feature artifact")
    model = RecipeEncoder(features.shape[1], saved["hidden"]).to(DEVICE)
    model.load_state_dict(saved["model"])
    model.eval()
    data = _graph(features, artifact["splits"]["train"])
    names = {row["id"]: row["name"] for row in unique_recipes(_read_csv(args.recipes))}
    if recipe_id not in names:
        raise ValueError(f"recipe ID absent from recipe CSV: {recipe_id}")

    with torch.no_grad():
        indices, similarities = rank_similar_recipes(
            model, model(data), id_to_index[recipe_id], args.top_k
        )
    ranked_ids = [recipe_ids[index] for index in indices.tolist()]
    missing_names = [
        candidate_id for candidate_id in ranked_ids if candidate_id not in names
    ]
    if missing_names:
        raise ValueError(
            f"recipe IDs absent from recipe CSV: {', '.join(missing_names)}"
        )
    print(f"기준 레시피: {recipe_id} | {names[recipe_id]}")
    print("순위\t레시피 ID\t레시피 이름\t예측 유사도(0~1)")
    for rank, (candidate_id, similarity) in enumerate(
        zip(ranked_ids, similarities.tolist(), strict=True), start=1
    ):
        print(f"{rank}\t{candidate_id}\t{names[candidate_id]}\t{similarity:.4f}")


# ===================== 1단계 실행 설정 =====================
# 위에서부터 순서대로 실행. 필요 없는 명령은 주석 처리 (빈 목록이면 함수 정의만)
STAGE1_RUN = [
    "prepare",  # BGE 임베딩 + train/val/test 쌍 분할 -> STAGE1_EMBEDDINGS
    "train",  # 학습 -> STAGE1_CHECKPOINT
    "evaluate",  # 검증/테스트 쌍 예측 -> STAGE1_PREDICTIONS_CSV
    "ranking-metrics",  # 예측 CSV의 NDCG/Precision/Recall@K, Spearman
    # "predict",  # 두 레시피(a_id, b_id)의 유사도
    # "top-similar",  # recipe_id와 유사한 상위 top_k 레시피
]

STAGE1_ARGS = {
    "prepare": argparse.Namespace(
        recipes=RECIPES_CSV, pairs=PAIRS_CSV, output=STAGE1_EMBEDDINGS,
        batch_size=32, seed=42,
    ),
    "train": argparse.Namespace(
        data=STAGE1_EMBEDDINGS, output=STAGE1_CHECKPOINT,
        hidden=32, epochs=100, folds=5, patience=10, lr=0.01, seed=42,
    ),
    "evaluate": argparse.Namespace(
        data=STAGE1_EMBEDDINGS, checkpoint=STAGE1_CHECKPOINT, output=STAGE1_PREDICTIONS_CSV,
    ),
    "ranking-metrics": argparse.Namespace(
        predictions=STAGE1_PREDICTIONS_CSV, k=list(DEFAULT_KS), threshold=DEFAULT_THRESHOLD,
        score_column="predicted_similarity", group_by="none",  # group_by: "none" | "recipe"
    ),
    "predict": argparse.Namespace(
        data=STAGE1_EMBEDDINGS, checkpoint=STAGE1_CHECKPOINT, a_id="172520", b_id="306863",
    ),
    "top-similar": argparse.Namespace(
        data=STAGE1_EMBEDDINGS, checkpoint=STAGE1_CHECKPOINT, recipes=RECIPES_CSV,
        recipe_id="172520", top_k=10,
    ),
}

STAGE1_COMMANDS = {
    "prepare": stage1_prepare,
    "train": stage1_train,
    "evaluate": stage1_evaluate,
    "ranking-metrics": ranking_metrics_command,
    "predict": stage1_predict,
    "top-similar": stage1_top_similar,
}
for command in STAGE1_RUN:
    print(f"\n===== stage1 {command} =====")
    STAGE1_COMMANDS[command](STAGE1_ARGS[command])

In [ ]:
# ======================================================================
# [5] 2단계: 재료 기여도 학습  (training/stage2.py)
# ======================================================================
from __future__ import annotations

# 2단계 레시피-재료 기여도 학습
#     - 실행 순서: prepare(재료 임베딩·기여도 분할 생성) → train


import argparse
import copy
from pathlib import Path

import torch
from torch.nn import functional as F


def _without_pairs(data, pairs):
    # 예측 대상 연결을 메시지 전달 그래프에서 뺀 얕은 사본 (노드 특징은 공유)
    store = data[CONTRIBUTION]
    width = data["Ingredient"].num_nodes
    keys = store.edge_index[0] * width + store.edge_index[1]
    keep = ~torch.isin(keys, pairs[0] * width + pairs[1])
    out = copy.copy(data)
    predicted = store.is_predicted[keep] if "is_predicted" in store else None
    set_contributions(out, store.edge_index[:, keep], store.edge_weight[keep], predicted)
    return out


def stage2_train(
    data,
    targets,
    recipe_model,
    *,
    epochs: int = 100,
    lr: float = 0.01,
    folds: int = 5,
    seed: int = 42,
):
    if epochs < 1 or lr <= 0 or folds < 1:
        raise ValueError("epochs, lr and folds must be positive")
    recipe_model.eval()
    for parameter in recipe_model.parameters():
        parameter.requires_grad_(False)
    with torch.no_grad():
        recipe_embeddings = recipe_model(data).detach()
    # 레시피 임베딩과 같은 장치(GPU/CPU)에서 학습
    device = recipe_embeddings.device
    model = IngredientLinkEncoder(
        data["Ingredient"].x.shape[1], recipe_embeddings.shape[1]
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    pairs, labels = targets["train"]["ingredient"]
    pairs, labels = pairs.to(device), labels.to(device)
    if labels.numel() == 0:
        raise ValueError("stage2 train split needs labeled contribution pairs")

    # 연결된 레시피가 없는 재료도 자체 특징으로 평가할 수 있도록 연결을 비운 feature_only 그래프를 함께 학습
    feature_only_data = _without_pairs(data, data[CONTRIBUTION].edge_index)
    generator = torch.Generator().manual_seed(seed)

    # 폴드 교대: 현재 폴드의 정답 연결은 입력 그래프에서 제외하고 나머지 연결로 예측
    for _ in range(epochs):
        model.train()
        order = torch.randperm(labels.numel(), generator=generator)
        for selected in order.chunk(min(folds, labels.numel())):
            selected = selected.to(device)
            fold_pairs, fold_labels = pairs[:, selected], labels[selected]
            context = _without_pairs(data, fold_pairs)
            optimizer.zero_grad()
            context_loss = F.mse_loss(
                model.score(context, recipe_embeddings, fold_pairs).sigmoid(),
                fold_labels,
            )
            feature_loss = F.mse_loss(
                model.score(feature_only_data, recipe_embeddings, fold_pairs).sigmoid(),
                fold_labels,
            )
            loss = (context_loss + feature_loss) / 2
            loss.backward()
            optimizer.step()

    return model, float(loss.detach())


def stage2_prepare(args: argparse.Namespace) -> None:

    encoder = BGETextEncoder()
    artifact = prepare_stage2(
        lambda texts: encoder.encode_texts(
            texts, batch_size=args.batch_size, show_progress_bar=True
        ),
        category_path=args.categories,
        contribution_path=args.contributions,
        seed=args.seed,
        test_ratio=args.test_ratio,
    )
    artifact["encoder"] = encoder.MODEL_NAME
    args.output.parent.mkdir(parents=True, exist_ok=True)
    torch.save(artifact, args.output)
    uncategorized = sum(not category for category in artifact["ingredient_categories"])
    print(
        f"saved {len(artifact['ingredient_names'])} ingredients "
        f"({uncategorized} without category) to {args.output}"
    )
    for name, split in artifact["splits"].items():
        print(f"{name}: {len(split)} contribution pairs")


def stage2_train_command(args: argparse.Namespace) -> None:
    if args.epochs < 1 or args.lr <= 0 or args.folds < 1:
        raise ValueError("epochs, lr and folds must be positive")
    data, targets = load_dataset(args.stage1_data, args.data)
    data = data.to(DEVICE)
    stage1 = torch.load(args.stage1_data, map_location="cpu", weights_only=True)
    saved = torch.load(args.stage1_checkpoint, map_location="cpu", weights_only=True)
    if saved["recipe_features"] != data["Recipe"].x.shape[1]:
        raise ValueError("recipe feature width differs from the stage-1 checkpoint")
    if "data_seed" in saved and saved["data_seed"] != stage1["seed"]:
        raise ValueError("stage-1 checkpoint does not match the stage-1 embeddings")
    recipe_model = RecipeEncoder(saved["recipe_features"], saved["hidden"]).to(DEVICE)
    recipe_model.load_state_dict(saved["model"])
    model, loss = stage2_train(
        data,
        targets,
        recipe_model,
        epochs=args.epochs,
        lr=args.lr,
        folds=args.folds,
        seed=args.seed,
    )
    checkpoint = {
        # GPU가 없는 환경에서도 읽을 수 있도록 CPU 텐서로 저장
        "model": {k: v.cpu() for k, v in model.state_dict().items()},
        "hidden": saved["hidden"],
        "ingredient_features": data["Ingredient"].x.shape[1],
    }
    args.output.parent.mkdir(parents=True, exist_ok=True)
    torch.save(checkpoint, args.output)
    print(f"stage=2 train_loss={loss:.4f} checkpoint={args.output}")


# ===================== 2단계 실행 설정 =====================
# 위에서부터 순서대로 실행. 필요 없는 명령은 주석 처리 (빈 목록이면 함수 정의만)
STAGE2_RUN = [
    "prepare",  # 재료 BGE 임베딩 + 기여도 train/test 분할 -> STAGE2_EMBEDDINGS
    "train",  # 1단계 체크포인트로 레시피 임베딩 고정 후 기여도 학습 -> STAGE2_CHECKPOINT
]

STAGE2_ARGS = {
    "prepare": argparse.Namespace(
        categories=CATEGORY_CSV, contributions=CONTRIBUTION_CSV, output=STAGE2_EMBEDDINGS,
        batch_size=32, test_ratio=0.15, seed=42,
    ),
    "train": argparse.Namespace(
        stage1_data=STAGE1_EMBEDDINGS, data=STAGE2_EMBEDDINGS,
        stage1_checkpoint=STAGE1_CHECKPOINT, output=STAGE2_CHECKPOINT,
        epochs=100, folds=5, lr=0.01, seed=42,
    ),
}

STAGE2_COMMANDS = {"prepare": stage2_prepare, "train": stage2_train_command}
for command in STAGE2_RUN:
    print(f"\n===== stage2 {command} =====")
    STAGE2_COMMANDS[command](STAGE2_ARGS[command])

In [ ]:
# ======================================================================
# [6] 대체 재료 기반 레시피 추천  (src/main.py)
# ======================================================================
from __future__ import annotations

# 재료 대체 기반 레시피 추천 콘솔 프로그램
#
# 실행: 아래 실행 설정 확인 후 셀 실행 (입력창에 q 입력 시 종료)
#
# 1. 레시피 ID 입력
# 2. 해당 레시피의 재료(기여도 CSV) 중 제외할 재료 선택
# 3. 같은 카테고리에서 기여도가 비슷한 대체 재료 후보 중 하나 선택
# 4. 대체 재료가 들어가면서 원래 레시피와 유사한 레시피 목록 출력


import argparse
import csv
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import torch
from torch import Tensor
from torch_geometric.data import HeteroData




class Quit(Exception):
    """사용자가 종료를 선택"""


@dataclass
class App:
    data: HeteroData
    recipe_embeddings: Tensor
    model: IngredientLinkEncoder
    recipe_index: dict[str, int]  # 원본 레시피 ID -> Recipe 노드 번호
    ingredient_index: dict[str, int]  # 재료 이름 -> Ingredient 노드 번호
    contributions: dict[
        str, list[tuple[str, float]]
    ]  # 원본 레시피 ID -> [(재료, 기여도)]
    recipe_names: dict[str, str]
    contribution_path: Path
    top_k: int

    def recipe_label(self, raw_id: str) -> str:
        return f"[{raw_id}] {self.recipe_names.get(raw_id, '(이름 없음)')}"


def _read_recipe_names(path: Path) -> dict[str, str]:
    csv.field_size_limit(10**9)  # steps 컬럼이 기본 한도보다 길 수 있음
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return {
            row["id"].strip(): row["name"].strip() for row in csv.DictReader(handle)
        }


def load_app(args: argparse.Namespace) -> App:
    required = {
        args.stage1_data: "[4] stage1 블록의 prepare",
        args.stage1_checkpoint: "[4] stage1 블록의 train",
        args.data: "[5] stage2 블록의 prepare",
        args.stage2_checkpoint: "[5] stage2 블록의 train",
        args.recipes: None,
        args.contributions: None,
    }
    missing = [
        (path, command) for path, command in required.items() if not path.exists()
    ]
    if missing:
        lines = ["실행에 필요한 파일이 없습니다:"]
        for path, command in missing:
            lines.append(f"  - {path}" + (f"  (생성: {command})" if command else ""))
        raise SystemExit("\n".join(lines))

    data, targets = load_dataset(args.stage1_data, args.data)
    # 추천할 때는 테스트 분할까지 포함한 실제 연결 전체를 그래프에 넣는다.
    # (학습 분할만 넣으면 레시피에 이미 들어 있는 재료가 대체 후보로 나올 수 있음)
    pairs = torch.cat(
        [targets[split]["ingredient"][0] for split in ("train", "test")], dim=1
    )
    weights = torch.cat(
        [targets[split]["ingredient"][1] for split in ("train", "test")]
    )
    set_contributions(data, pairs, weights)
    data = data.to(DEVICE)

    stage1 = torch.load(args.stage1_checkpoint, map_location="cpu", weights_only=True)
    stage2 = torch.load(args.stage2_checkpoint, map_location="cpu", weights_only=True)
    if (
        stage1["recipe_features"] != data["Recipe"].x.shape[1]
        or stage2["ingredient_features"] != data["Ingredient"].x.shape[1]
    ):
        raise SystemExit("체크포인트와 데이터의 특징 차원이 맞지 않습니다")
    if stage1["hidden"] != stage2["hidden"]:
        raise SystemExit("1단계와 2단계 체크포인트의 hidden 크기가 다릅니다")
    recipe_model = RecipeEncoder(stage1["recipe_features"], stage1["hidden"]).to(DEVICE)
    model = IngredientLinkEncoder(stage2["ingredient_features"], stage2["hidden"]).to(DEVICE)
    recipe_model.load_state_dict(stage1["model"])
    model.load_state_dict(stage2["model"])
    recipe_model.eval()
    model.eval()
    with torch.no_grad():
        recipe_embeddings = recipe_model(data)

    contributions: dict[str, list[tuple[str, float]]] = defaultdict(list)
    for recipe_id, name, weight in read_contributions(args.contributions):
        contributions[recipe_id].append((name, weight))
    for rows in contributions.values():
        rows.sort(key=lambda row: -row[1])

    return App(
        data=data,
        recipe_embeddings=recipe_embeddings,
        model=model,
        recipe_index={rid: i for i, rid in enumerate(data["Recipe"].raw_id)},
        ingredient_index={
            name: i for i, name in enumerate(data["Ingredient"].ingredient_name)
        },
        contributions=dict(contributions),
        recipe_names=_read_recipe_names(args.recipes),
        contribution_path=args.contributions,
        top_k=args.top_k,
    )


def _ask(prompt: str) -> str:
    try:
        answer = input(prompt).strip()
    except EOFError:
        raise Quit
    if answer.lower() in {"q", "quit", "exit"}:
        raise Quit
    return answer


def _choose(prompt: str, names: list[str]) -> str | None:
    """번호(1부터) 또는 이름으로 하나를 고른다. 빈 입력이면 None(처음으로)"""
    while True:
        answer = _ask(prompt)
        if not answer:
            return None
        if answer.isdigit() and 1 <= int(answer) <= len(names):
            return names[int(answer) - 1]
        if answer.lower() in names:
            return answer.lower()
        print(f"  1~{len(names)} 사이의 번호나 목록에 있는 이름을 입력하세요.")


def find_substitutes(
    app: App, recipe_id: str, ingredient: str, contribution: float
) -> list[tuple[str, float]]:
    """제외할 재료와 같은 카테고리에서 기여도가 비슷한 재료 [(이름, 예측 기여도)]"""
    ids, weights = app.model.recommend_substitutes(
        app.data,
        app.recipe_embeddings,
        recipe_id=app.recipe_index[recipe_id],
        ingredient_id=app.ingredient_index[ingredient],
        top_k=app.top_k,
        original_contribution=contribution,
    )
    names = app.data["Ingredient"].ingredient_name
    return [(names[i], weight) for i, weight in zip(ids.tolist(), weights.tolist())]


def find_recipes(app: App, recipe_id: str, ingredient: str) -> list[tuple[str, float]]:
    """ingredient가 들어가는 레시피 중 recipe_id와 유사한 순 [(원본 레시피 ID, 유사도)]"""
    ids, similarities = app.model.recommend_recipes(
        app.data,
        app.recipe_embeddings,
        ingredient_id=app.ingredient_index[ingredient],
        recipe_id=app.recipe_index[recipe_id],
        top_k=app.top_k,
        contribution_path=app.contribution_path,
    )
    raw_ids = app.data["Recipe"].raw_id
    return [
        (raw_ids[i], score) for i, score in zip(ids.tolist(), similarities.tolist())
    ]


def run_once(app: App) -> None:
    # 1. 레시피 선택
    recipe_id = _ask("\n레시피 ID를 입력하세요 (종료: q): ")
    if not recipe_id:
        return
    if recipe_id not in app.recipe_index:
        print("  그래프에 없는 레시피 ID입니다.")
        return
    ingredients = app.contributions.get(recipe_id)
    if not ingredients:
        print(
            f"  {app.recipe_label(recipe_id)}: 기여도 데이터에 재료 정보가 없는 레시피입니다."
        )
        return

    # 2. 제외할 재료 선택
    print(f"\n{app.recipe_label(recipe_id)} 의 재료")
    for number, (name, weight) in enumerate(ingredients, 1):
        print(f"  {number:2}. {name}  (기여도 {weight:.3f})")
    while True:
        removed = _choose(
            "제외할 재료의 번호나 이름 (처음으로: Enter): ", [n for n, _ in ingredients]
        )
        if removed is None:
            return
        try:
            substitutes = find_substitutes(
                app, recipe_id, removed, dict(ingredients)[removed]
            )
        except ValueError:
            # 카테고리 CSV에 없는 재료 (예: salt and pepper, 비식재료)
            print(
                f"  '{removed}'은(는) 카테고리 정보가 없어 대체 재료를 찾을 수 없습니다."
            )
            continue
        if not substitutes:
            print(f"  '{removed}'과(와) 같은 카테고리의 대체 후보가 없습니다.")
            continue
        break

    # 3. 대체 재료 선택
    print(
        f"\n'{removed}' 대신 쓸 수 있는 재료 (기여도 {dict(ingredients)[removed]:.3f}와 가까운 순)"
    )
    for number, (name, weight) in enumerate(substitutes, 1):
        print(f"  {number:2}. {name}  (예측 기여도 {weight:.3f})")
    substitute = _choose(
        "대체 재료의 번호나 이름 (처음으로: Enter): ", [n for n, _ in substitutes]
    )
    if substitute is None:
        return

    # 4. 대체 재료가 들어가는 유사 레시피
    recipes = find_recipes(app, recipe_id, substitute)
    if not recipes:
        print(f"\n'{substitute}'이(가) 들어가는 다른 레시피가 없습니다.")
        return
    print(
        f"\n'{substitute}'이(가) 들어가고 {app.recipe_label(recipe_id)} 와 유사한 레시피"
    )
    for number, (raw_id, score) in enumerate(recipes, 1):
        print(f"  {number:2}. {app.recipe_label(raw_id)}  (유사도 {score:.3f})")


def main(args: argparse.Namespace) -> None:
    if args.top_k < 1:
        raise ValueError("top_k는 1 이상이어야 합니다")

    print("모델과 데이터를 불러오는 중...")
    app = load_app(args)
    print(
        f"레시피 {len(app.recipe_index)}개, 재료 정보가 있는 레시피 {len(app.contributions)}개"
    )
    try:
        while True:
            run_once(app)
    except (Quit, KeyboardInterrupt):
        print("\n종료합니다.")


# ===================== 실행 설정 =====================
MAIN_ARGS = argparse.Namespace(
    stage1_data=STAGE1_EMBEDDINGS,
    data=STAGE2_EMBEDDINGS,
    stage1_checkpoint=STAGE1_CHECKPOINT,
    stage2_checkpoint=STAGE2_CHECKPOINT,
    recipes=RECIPES_CSV,
    contributions=CONTRIBUTION_CSV,
    top_k=10,  # 후보로 보여줄 개수
)
RUN_MAIN = True  # False면 함수 정의만

if RUN_MAIN:
    main(MAIN_ARGS)